In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

import numpy as np
import osmnx as ox
import pandas as pd

np.random.default_rng(0)
ox.__version__

In [ ]:
place = "Isle of Wight, England"

In [ ]:
G = ox.graph_from_place(place, network_type="drive")
Gp = ox.project_graph(G)

In [ ]:
ox.plot_graph_folium(G);

In [ ]:
G = ox.routing.add_edge_speeds(G)  # for shortest path by travel time.
G = ox.routing.add_edge_travel_times(G)

G = ox.bearing.add_edge_bearings(G)  # for text directions.

In [ ]:
ox.geocode("Ryde, Isle of Wight")

In [ ]:
orig = ox.distance.nearest_nodes(G, X=-1.1633, Y=50.7300)
dest = ox.distance.nearest_nodes(G, X=-1.5397, Y=50.6795)

In [ ]:
# find the shortest path between nodes, minimizing travel time, then plot it.
route = ox.shortest_path(G, orig, dest, weight="travel_time")
fig, ax = ox.plot_graph_route(G, route, node_size=0)

In [ ]:
route

In [ ]:
# how long is our route in meters?
edge_lengths = ox.routing.route_to_gdf(G, route)["length"]
round(sum(edge_lengths))

In [ ]:
route_df = ox.routing.route_to_gdf(G, route)

In [ ]:
len(route_df)

In [ ]:
route_df.dtypes

# Generate Text Directions

One Approach to serialize this geojson data.

In [ ]:
# Impute NaN in 'name' as "Unnamed Road"
route_df["name"] = route_df["name"].apply(
    lambda x: x if not isinstance(x, list) and pd.notna(x) else "Unnamed Road",
)

In [ ]:
# Add a previous bearing column
route_df["prev_bearing"] = route_df["bearing"].shift(1)

In [ ]:
def bearing_to_compass(
    bearing: float,
) -> Literal[
    "North",
    "North East",
    "East",
    "South East",
    "South",
    "South West",
    "West",
    "North West",
]:
    """Convert bearing in degrees to an eight-way compass direction."""
    directions = [
        "North",
        "North East",
        "East",
        "South East",
        "South",
        "South West",
        "West",
        "North West",
    ]
    idx = round(bearing / 45) % 8
    return directions[idx]

In [ ]:
dir_continue = "Continue on"
def determine_turn(prev_bearing: float, curr_bearing: float) -> str:
    """Create the verb used in text directions."""
    return_verb_string: str
    if pd.isna(prev_bearing):
        return "Start on"
    change = (curr_bearing - prev_bearing + 360) % 360
    smallish_direction_change = 30
    smallish_direction_change_other_way = 330
    median_line = 180
    if change < smallish_direction_change or change > smallish_direction_change_other_way:
        return_verb_string = dir_continue
    elif change < median_line:
        return_verb_string = "Turn right onto"
    else:
        return_verb_string = "Turn left onto"
    return return_verb_string

In [ ]:
@dataclass
class TextGuidanceStep:
    """A textual driving description of one or multiple line segment(s) of the journey."""

    turn: Literal[
        "Start on",
        "Continue on",
        "Turn right onto",
        "Turn left onto",
        "Enter the roundabout and take the exit onto",
    ]
    name: str | list[str]
    compass_direction: Literal[
        "North",
        "North East",
        "East",
        "South East",
        "South",
        "South West",
        "West",
        "North West",
    ]
    length: float
    bridge: bool = False

    def __repr__(self):
        """Pretty Print guidance as a string."""
        # Handle street name being a list
        street_name = " aka ".join(self.name) if isinstance(self.name, list) else self.name

        # Emojis for turn directions
        turn_emojis = {
            "Start on": "🌅",
            "Continue on": "•",
            "Turn right onto": "↱",
            "Turn left onto": "↰",
            "Enter the roundabout and take the exit onto": "🔄",
        }

        # Format length in SI notation
        metres_per_kilometer = 1000
        if self.length < metres_per_kilometer:
            length_str = f"{round(self.length / 10) * 10} metres"
        else:
            length_str = f"{round(self.length / 100) / 10:.1f} kilometres"

        # Create the textual representation
        return (
            f"{turn_emojis[self.turn]} {self.turn} {street_name}"
            f" and head {self.compass_direction} for {length_str}"
            f"{' on the bridge 🌉' if self.bridge else ''}."
        )

In [ ]:
def generate_text_guidance_step(row) -> TextGuidanceStep:
    """Generate Text Guidance for each route linesting."""
    turn_instruction = determine_turn(row["prev_bearing"], row["bearing"])

    compass_dir = bearing_to_compass(row["bearing"])
    bridge = pd.notna(row["bridge"])

    return TextGuidanceStep(
        turn=turn_instruction,
        name=row["name"],
        compass_direction=compass_dir,
        length=row["length"],
        bridge=bridge,
    )

In [ ]:
route_df["text_guidance"] = route_df.apply(generate_text_guidance_step, axis=1)  # type: ignore  # noqa: PGH003

In [ ]:
route_df

In [ ]:
def is_collapsible(step1: TextGuidanceStep, step2: TextGuidanceStep) -> bool:
    """
    Boolean Test used to simplify the generated route.

    Determines whether two TextGuidanceStep instances are collapsible
    based on their turn instructions and names.

    Before Accumulation, a TextGuidanceStep corresponds to one linestring.
    Accumulated guidance aims to only provide driver-relevant information as needed.
    """
    return (step2.turn in [dir_continue]) and step1.name == step2.name

In [ ]:
def is_collapsible_loose(step1: TextGuidanceStep, step2: TextGuidanceStep) -> bool:
    """
    Boolean Test used to simplify the generated route.

    Determines if two TextGuidanceStep instances are collapsible based on their street names.
    This will return fewer or equal text instruction steps than is_collapsible func.
    """
    return step1.name == step2.name

In [ ]:
# Accumulate Rows
def accumulate_guidance(
    df: pd.DataFrame,
    collapsability_func: callable = is_collapsible_loose,  # type: ignore  # noqa: PGH003
) -> list[str]:
    """
    Accumulates TextGuidanceStep instances based on a collapsibility function.

    The dataframe's "text_guidance" column provides linestring by linestring directions.
    The output provides shorter human-relevant driving directions.

    Args:
        df: The DataFrame containing TextGuidanceStep instances.
        collapsibility_func: The function used to determine if two steps are collapsible.

    Returns:
        A list of accumulated TextGuidanceStep instances.
    """
    accumulated_steps = []
    current_step = None

    for _, row in df.iterrows():
        if current_step is None:
            current_step = row["text_guidance"]
        elif collapsability_func(current_step, row["text_guidance"]):
            current_step.length += row["text_guidance"].length
        else:
            accumulated_steps.append(current_step)
            current_step = row["text_guidance"]

    if current_step is not None:
        accumulated_steps.append(current_step)

    # Prepends the index to each string
    numbered_accumulated_steps = []
    for index, item in enumerate(accumulated_steps, start=1):
        numbered_accumulated_steps.append(f"{index}. {item}")

    return numbered_accumulated_steps

In [ ]:
longer_accumulated_guidance = accumulate_guidance(route_df, collapsability_func=is_collapsible)

In [ ]:
# Apply the first pass to accumulate the guidance
accumulated_guidance = accumulate_guidance(route_df)

In [ ]:
# before accumulation.
len(route_df["text_guidance"].to_numpy().tolist()), route_df["text_guidance"].to_numpy().tolist()

In [ ]:
# lighter accumulation.
len(longer_accumulated_guidance), longer_accumulated_guidance

In [ ]:
# full accumulation.
len(accumulated_guidance), accumulated_guidance